# Epi Info AI stratified 2 x 2 validation lab — V0.8

Compare deployed Rust/WebAssembly Mantel-Haenszel, OR/RR homogeneity, and exact adjusted-OR methods with independent Python formulas. Passing is evidence, not statistical approval.

In [ ]:
import math
from pyodide.http import pyfetch
from js import WebAssembly, Uint8Array
fixture_response = await pyfetch('/validation-fixtures/stratified-two-by-two-v0.5.json')
fixture_response.raise_for_status(); fixture = await fixture_response.json()
wasm_response = await pyfetch('/epi2x2.wasm')
instance = await WebAssembly.instantiate(Uint8Array.new(await wasm_response.arrayBuffer()), {})
rust = instance.instance.exports
for i, s in enumerate(fixture['input']['strata']):
    assert rust.stratified_set_table(i, s['exposedCases'], s['exposedNonCases'], s['unexposedCases'], s['unexposedNonCases']) == 1
n = len(fixture['input']['strata'])
rust_results = {'adjustedOddsRatio': float(rust.stratified_mh_odds_ratio(n)), 'adjustedRiskRatio': float(rust.stratified_mh_risk_ratio(n)), 'mantelHaenszelUncorrected': float(rust.stratified_mh_chi_square_uncorrected(n)), 'mantelHaenszelCorrected': float(rust.stratified_mh_chi_square_corrected(n))}
rust_results

In [ ]:
import csv, hashlib, io
data_response = await pyfetch('/examples/foodborne-outbreak-investigation.csv')
data_response.raise_for_status(); data_bytes = await data_response.bytes()
assert hashlib.sha256(data_bytes).hexdigest() == fixture['provenance']['datasetSha256']
records = list(csv.DictReader(io.StringIO(data_bytes.decode('utf-8-sig'))))
case_values = {'Confirmed', 'Probable', 'Suspected'}; derived = {}
for record in records:
    cells = derived.setdefault(record['Sex'], [0, 0, 0, 0]); exposed = record['Potato Salad'] == 'Yes'; case = record['Case Status'] in case_values
    cells[(0 if exposed else 2) + (0 if case else 1)] += 1
expected_tables = {s['label']: [s['exposedCases'], s['exposedNonCases'], s['unexposedCases'], s['unexposedNonCases']] for s in fixture['input']['strata']}
assert derived == expected_tables
derived

In [ ]:
tables = [(s['exposedCases'], s['exposedNonCases'], s['unexposedCases'], s['unexposedNonCases']) for s in fixture['input']['strata']]
or_num = sum(a*d/(a+b+c+d) for a,b,c,d in tables); or_den = sum(b*c/(a+b+c+d) for a,b,c,d in tables)
rr_num = sum(a*(c+d)/(a+b+c+d) for a,b,c,d in tables); rr_den = sum(c*(a+b)/(a+b+c+d) for a,b,c,d in tables)
mh_num = sum((a*d-b*c)/(a+b+c+d) for a,b,c,d in tables)
mh_den = sum((a+b)*(c+d)*(a+c)*(b+d)/((a+b+c+d-1)*(a+b+c+d)**2) for a,b,c,d in tables)
python_results = {'adjustedOddsRatio': or_num/or_den, 'adjustedRiskRatio': rr_num/rr_den, 'mantelHaenszelUncorrected': mh_num**2/mh_den, 'mantelHaenszelCorrected': max(abs(mh_num)-0.5, 0)**2/mh_den}
for name, expected in fixture['expected'].items():
    if name in rust_results:
        assert math.isclose(rust_results[name], expected, abs_tol=fixture['tolerance'], rel_tol=0)
        assert math.isclose(python_results[name], expected, abs_tol=fixture['tolerance'], rel_tol=0)
print('PASS: foodborne stratified anchors')

## OR/RR homogeneity

The standard fixed-margin Breslow-Day statistic, Tarone correction, and the Woolf OR/RR statistics shown under familiar legacy Epi Info Breslow-Day labels are validated separately.

In [ ]:
hom_response = await pyfetch('/validation-fixtures/stratified-homogeneity-v0.7.json')
hom_response.raise_for_status(); hom = await hom_response.json()
for i, s in enumerate(hom['input']['strata']):
    assert rust.stratified_set_table(i, s['exposedCases'], s['exposedNonCases'], s['unexposedCases'], s['unexposedNonCases']) == 1
k = len(hom['input']['strata']); df = k - 1
candidate = {'breslowDayOddsRatio': float(rust.stratified_breslow_day_odds_ratio(k)), 'breslowDayTaroneOddsRatio': float(rust.stratified_breslow_day_tarone_odds_ratio(k)), 'legacyBreslowDayOddsRatio': float(rust.stratified_legacy_woolf_odds_ratio(k)), 'legacyBreslowDayRiskRatio': float(rust.stratified_legacy_woolf_risk_ratio(k))}
for name, value in candidate.items():
    assert math.isclose(value, hom['expected'][name], abs_tol=hom['tolerance'], rel_tol=0)
    assert math.isclose(float(rust.chi_square_p_value_df(value, df)), hom['expected'][name + 'PValue'], abs_tol=hom['tolerance'], rel_tol=0)
candidate

In [ ]:
from scipy.stats import chi2
htables = [(s['exposedCases'], s['exposedNonCases'], s['unexposedCases'], s['unexposedNonCases']) for s in hom['input']['strata']]
common_or = sum(a*d/(a+b+c+d) for a,b,c,d in htables) / sum(b*c/(a+b+c+d) for a,b,c,d in htables)
bd = expected_delta = variance_sum = 0.0
for a,b,c,d in htables:
    exposed, cases, total = a+b, a+c, a+b+c+d; lo, hi = max(0.0, exposed-(b+d)), min(exposed, cases)
    for _ in range(160):
        x = (lo+hi)/2; odds = x*(total-exposed-cases+x)/((exposed-x)*(cases-x))
        if odds < common_or: lo = x
        else: hi = x
    expected = (lo+hi)/2; cells = (expected, exposed-expected, cases-expected, total-exposed-cases+expected); variance = 1/sum(1/cell for cell in cells)
    bd += (a-expected)**2/variance; expected_delta += a-expected; variance_sum += variance
tarone = bd - expected_delta**2/variance_sum
weights = [1/(1/a+1/b+1/c+1/d) for a,b,c,d in htables]; logs = [math.log(a*d/(b*c)) for a,b,c,d in htables]
pooled = sum(w*x for w,x in zip(weights,logs))/sum(weights); woolf = sum(w*(x-pooled)**2 for w,x in zip(weights,logs))
rr = [(a/(a+b))/(c/(c+d)) for a,b,c,d in htables]
rr_weights = [1/(b/(a*(a+b))+d/(c*(c+d))) for a,b,c,d in htables]
pooled_rr = sum(w*math.log(x) for w,x in zip(rr_weights,rr))/sum(rr_weights)
woolf_rr = sum(w*(math.log(x)-pooled_rr)**2 for w,x in zip(rr_weights,rr))
independent = {'breslowDayOddsRatio': bd, 'breslowDayTaroneOddsRatio': tarone, 'legacyBreslowDayOddsRatio': woolf, 'legacyBreslowDayRiskRatio': woolf_rr}
for name, value in independent.items():
    assert math.isclose(value, hom['expected'][name], abs_tol=hom['tolerance'], rel_tol=0)
    assert math.isclose(float(chi2.sf(value, df)), hom['expected'][name + 'PValue'], abs_tol=hom['tolerance'], rel_tol=0)
print('PASS: Rust/WASM, direct Python formulas, SciPy p-values, and V0.7 anchors agree')

## Exact adjusted odds ratio — V0.8

The candidate CMLE and central Fisher limits use the product-hypergeometric distribution conditioned on every stratum's margins.

In [ ]:
exact_response = await pyfetch('/validation-fixtures/stratified-exact-v0.8.json')
exact_response.raise_for_status(); exact_fixture = await exact_response.json()
for case in exact_fixture['cases']:
    for i, s in enumerate(case['input']['strata']):
        assert rust.stratified_set_table(i, s['exposedCases'], s['exposedNonCases'], s['unexposedCases'], s['unexposedNonCases']) == 1
    count = len(case['input']['strata']); expected = case['expected']; level = case['input']['confidenceLevel']
    candidate = {'estimate': float(rust.stratified_conditional_odds_ratio(count)), 'lower': float(rust.stratified_conditional_odds_ratio_fisher_lower(count, level)), 'upper': float(rust.stratified_conditional_odds_ratio_fisher_upper(count, level))}
    for name, value in candidate.items(): assert math.isclose(value, expected[name], abs_tol=exact_fixture['tolerance'], rel_tol=0)
candidate

In [ ]:
import numpy as np
from scipy.optimize import brentq
from scipy.special import gammaln, logsumexp
def exact_distribution(tables, log_odds):
    coefficients = np.array([1.0]); offset = 0
    for a,b,c,d in tables:
        cases, noncases, exposed = a+c, b+d, a+b; lower, upper = max(0, exposed-noncases), min(cases, exposed); offset += lower
        logs = np.array([gammaln(cases+1)-gammaln(x+1)-gammaln(cases-x+1)+gammaln(noncases+1)-gammaln(exposed-x+1)-gammaln(noncases-exposed+x+1) for x in range(lower, upper+1)])
        coefficients = np.convolve(coefficients, np.exp(logs-logs.max())); coefficients /= coefficients.max()
    support = np.arange(offset, offset+len(coefficients)); weights = np.log(coefficients)+support*log_odds; probabilities = np.exp(weights-logsumexp(weights))
    return support, probabilities
for case in exact_fixture['cases']:
    tables = [(s['exposedCases'],s['exposedNonCases'],s['unexposedCases'],s['unexposedNonCases']) for s in case['input']['strata']]; observed = sum(t[0] for t in tables); alpha = (1-case['input']['confidenceLevel'])/2
    mean = lambda z: sum(np.prod(pair) for pair in zip(*exact_distribution(tables,z)))
    estimate = math.exp(brentq(lambda z: mean(z)-observed, -700, 700))
    lower = math.exp(brentq(lambda z: exact_distribution(tables,z)[1][exact_distribution(tables,z)[0]>=observed].sum()-alpha, -700, 700))
    upper = math.exp(brentq(lambda z: exact_distribution(tables,z)[1][exact_distribution(tables,z)[0]<=observed].sum()-alpha, -700, 700))
    for name,value in {'estimate':estimate,'lower':lower,'upper':upper}.items(): assert math.isclose(value, case['expected'][name], abs_tol=exact_fixture['tolerance'], rel_tol=0)
print('PASS: Rust/WASM and independent SciPy-based product-hypergeometric inversion match V0.8 anchors')

## Operational and pathological evidence

These checks exercise explicit boundary states, the reviewed support/work limits, the 1,024-strata ceiling, and an intentionally generous CI-runner performance budget. G5 approval is deferred to the consolidated review of all outputs.

In [ ]:
import time
operational_response = await pyfetch('/validation-fixtures/stratified-operational-v0.8.json')
operational_response.raise_for_status(); operational = await operational_response.json()
def state(value):
    return 'unavailable' if math.isnan(value) else ('positive-infinity' if math.isinf(value) and value > 0 else ('zero' if value == 0 else 'finite'))
for case in operational['literalCases']:
    for i, s in enumerate(case['input']['strata']):
        assert rust.stratified_set_table(i, s['exposedCases'], s['exposedNonCases'], s['unexposedCases'], s['unexposedNonCases']) == 1
    count = len(case['input']['strata']); level = case['input']['confidenceLevel']; expected = case['expected']
    assert state(float(rust.stratified_conditional_odds_ratio(count))) == expected['estimateState']
    assert state(float(rust.stratified_conditional_odds_ratio_fisher_lower(count, level))) == expected['lowerState']
    assert state(float(rust.stratified_conditional_odds_ratio_fisher_upper(count, level))) == expected['upperState']
generated = operational['generatedCases'][0]; a,b,c,d = generated['repeatedCells']
for i in range(generated['strata']): assert rust.stratified_set_table(i, a, b, c, d) == 1
started = time.perf_counter(); mh_or = float(rust.stratified_mh_odds_ratio(generated['strata'])); exact = float(rust.stratified_conditional_odds_ratio(generated['strata'])); elapsed_ms = (time.perf_counter()-started)*1000
assert math.isclose(mh_or, generated['expected']['adjustedOddsRatio'], abs_tol=1e-12, rel_tol=0)
assert state(exact) == generated['expected']['exactState']
assert elapsed_ms <= generated['budget']['ciRunnerMilliseconds']
print(f'PASS: pathological states and 1,024-strata operational limit ({elapsed_ms:.1f} ms)')